Part1 层和块

1.1 自定义层

很灵活可以自己调用

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

# 生成 2 X 20 的随机矩阵作为输入 2是批量大小，20是维度和上面一致
X = torch.rand(2, 20)  
net(X) 

tensor([[-0.3555, -0.1269,  0.1272, -0.0825,  0.2825,  0.1437,  0.0424,  0.1437,
          0.0203, -0.0329],
        [-0.3196, -0.1268,  0.1820, -0.2715,  0.2043,  0.0210, -0.1387,  0.0788,
          0.0846,  0.0122]], grad_fn=<AddmmBackward0>)

1.2 自定义块

In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)
    
    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

1.3 实例化多层感知机的层，然后在每次调用正向传播函数时，调用这些层

In [3]:
net = MLP()
net(X)

tensor([[-0.0286, -0.1372, -0.4313, -0.1097, -0.2716,  0.2614,  0.1343,  0.2208,
         -0.1795, -0.2599],
        [-0.1005, -0.2647, -0.4365, -0.1788, -0.1718,  0.2607,  0.2763,  0.3226,
         -0.1661, -0.3804]], grad_fn=<AddmmBackward0>)

1.4 顺序块

_modules 特殊容器。会把传入的那些层，按顺序放入这个容器中

In [7]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for block in args:
            self._modules[block] = block
            
    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.2436, -0.1122,  0.1692,  0.1495,  0.1780, -0.0475, -0.1633,  0.1095,
          0.0046,  0.2048],
        [-0.3125, -0.2226,  0.1345,  0.0668,  0.0337, -0.0532, -0.1523,  0.0926,
          0.0906,  0.1961]], grad_fn=<AddmmBackward0>)

1.5 在正向传播函数中执行代码

这样来构造函数，可以在forward和 __init__ 中做自定义计算

这样我们自己操作会更灵活

In [9]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)
        
    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()
 
net = FixedHiddenMLP()
net(X)

tensor(-0.1260, grad_fn=<SumBackward0>)

1.6 混合搭配各种组合块的方法

In [12]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)
        
    def forward(self, X):
        return self.linear(self.net(X))
    
mix = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
mix(X)

tensor(0.0963, grad_fn=<SumBackward0>)